# baseline v5 - Gemma 4 31B-it **noise-robust** LoRA for VQA MCQA

이 노트북은 **원본 이미지를 직접 크롭/다운스케일하지 않고** PIL 원본을 그대로 `processor`에 넣는 전제를 유지하면서,
작은 규모(약 5천장), **라벨 노이즈/애매한 문항이 섞인** 객관식 VQA 데이터셋에 맞춰 **정확도 중심**으로 다시 짠 버전입니다.

핵심 변경점
- **200개 epoch subsampling 제거** → 매 epoch 전체 train pool 사용
- **랜덤 1-token 생성 loss** 중심에서 → **객관식 4지선다 choice-logit 학습**으로 변경
- Gemma 4의 **비추론(empty thought channel) 템플릿 이슈**를 피하기 위해  
  **dummy assistant answer로 정답 위치를 정렬한 뒤 그 위치의 4-choice logits만 학습**
- **강한 zero-shot teacher(base model) 기반 노이즈 추정**
  - teacher 확률 분포 캐시
  - 의심 샘플 down-weight
  - gold one-hot + teacher soft target 혼합
- **보기 순서 셔플 augmentation** 적용
- **valid loss 대신 clean-valid accuracy 중심** early stopping
- **추론 시 prompt ensemble + 보기 순서 ensemble + base/adaptor logit blending** 옵션 제공

권장 사용 순서
1. 환경 셀 실행 후 런타임 재시작
2. 데이터 압축 해제
3. teacher cache 생성 셀 한 번 실행
4. 학습
5. best adapter로 inference


# 환경 준비

- 아래 셀 실행
- 실행 후 **런타임/세션 다시 시작**
- Hugging Face에서 **Gemma 라이선스 동의**
- Gemma 4는 최신 `transformers` / `peft`가 필요한 경우가 있어 git main 설치 유지


In [ ]:

!pip -q install --upgrade --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install --upgrade git+https://github.com/huggingface/transformers.git
!pip -q install --upgrade git+https://github.com/huggingface/peft.git
!pip -q install --upgrade accelerate bitsandbytes datasets pillow pandas numpy sentencepiece protobuf

# 선택: A100/L4에서 flash attention을 쓰고 싶으면 주석 해제
!pip -q install flash-attn --no-build-isolation


In [ ]:

import torch, transformers, peft, bitsandbytes as bnb, numpy as np, pandas as pd
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("Transformers version:", transformers.__version__)
print("PEFT version:", peft.__version__)
print("BitsAndBytes version:", bnb.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))


# 데이터 준비

필수 파일
- `train.csv`, `train/`
- `test.csv`, `test/`
- `sample_submission.csv`

Colab + Google Drive 기준 예시입니다.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

!unzip "/content/drive/My Drive/2026-ssafy-ai-15-2.zip" -d "/content/"


# 설정, 공통 함수, 데이터 split

이 셀에서 다음을 모두 정의합니다.

- 하이퍼파라미터
- prompt template
- 보기 순서 permutation / remapping
- Gemma 4 chat template wrapper
- train/valid fixed split
- dataset / collator
- choice-logit 계산 함수
- teacher noise scoring / evaluation 보조 함수


In [ ]:

import os, re, gc, math, json, random, shutil, hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn.functional as F

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

try:
    from transformers import AutoModelForImageTextToText as AutoVisionTextModel
except ImportError:
    from transformers import AutoModelForMultimodalLM as AutoVisionTextModel

from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)

try:
    from peft import replace_lora_weights_loftq
except Exception:
    replace_lora_weights_loftq = None

from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"
Image.MAX_IMAGE_PIXELS = None

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---------------------------------
# 기본 설정
# ---------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = "google/gemma-4-31B-it"
ENABLE_THINKING = False

RUN_DIR = "/content/gemma4_31b_it_mcqa_noise_robust_lora"
RESUME_FROM_CHECKPOINT = None
# 예: RESUME_FROM_CHECKPOINT = "/content/gemma4_31b_it_mcqa_noise_robust_lora/checkpoint-epoch-03"

VALID_RATIO = 0.08

# training
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

NUM_EPOCHS_STAGE1 = 1   # cleaner subset warm-up
NUM_EPOCHS_STAGE2 = 4   # full weighted fine-tuning

EARLY_STOP_PATIENCE = 2
EARLY_STOP_MIN_DELTA = 1e-4

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
USE_RSLORA = True
USE_LOFTQ_INIT = replace_lora_weights_loftq is not None
ENABLE_VISION_TOWER_LORA = False

# inference / evaluation
PROMPT_VARIANT_IDS_TRAIN = [0, 1]
PROMPT_VARIANT_IDS_EVAL = [0, 1]
PROMPT_VARIANT_IDS_INFER = [0, 1]
TRAIN_OPTION_SHUFFLE_PROB = 1.0
INFER_NUM_OPTION_PERMUTATIONS = 4
INFER_VARIANT_BATCH_SIZE = 4
ENABLE_BASE_BLEND = True
BASE_BLEND_WEIGHT = 0.35   # final = (1-w) * adapter + w * base

# noisy-label robust target
LABEL_SMOOTHING = 0.02
TEACHER_BATCH_SIZE = 1
DUMMY_ASSISTANT_ANSWER = "a"

# 기타
NUM_WORKERS = 0
MAX_NEW_TOKENS = 4  # generation fallback/debug용
ATTN_IMPLEMENTATION = "eager"  # flash_attention_2 설치 시 변경 가능

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

os.makedirs(RUN_DIR, exist_ok=True)

# ---------------------------------
# 데이터 로드 + fixed split
# ---------------------------------
raw_train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

if "id" not in raw_train_df.columns:
    raw_train_df = raw_train_df.copy()
    raw_train_df["id"] = np.arange(len(raw_train_df))

if "id" not in test_df.columns:
    test_df = test_df.copy()
    test_df["id"] = np.arange(len(test_df))

train_pool_path = os.path.join(RUN_DIR, "train_pool_fixed.csv")
valid_path = os.path.join(RUN_DIR, "valid_fixed.csv")

if os.path.exists(train_pool_path) and os.path.exists(valid_path):
    train_pool_df = pd.read_csv(train_pool_path)
    valid_df = pd.read_csv(valid_path)
    print("고정 split 불러옴:", RUN_DIR)
else:
    shuffled_df = raw_train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    valid_size = max(1, int(len(shuffled_df) * VALID_RATIO))
    valid_df = shuffled_df.iloc[:valid_size].reset_index(drop=True)
    train_pool_df = shuffled_df.iloc[valid_size:].reset_index(drop=True)

    train_pool_df.to_csv(train_pool_path, index=False)
    valid_df.to_csv(valid_path, index=False)
    print("고정 split 저장:", RUN_DIR)

print("MODEL_ID:", MODEL_ID)
print("USE_BF16:", USE_BF16)
print("train_pool size:", len(train_pool_df))
print("valid size:", len(valid_df))
print("test size:", len(test_df))

# ---------------------------------
# 공통 상수 / prompt
# ---------------------------------
LETTERS = ["a", "b", "c", "d"]
LETTER_TO_IDX = {c: i for i, c in enumerate(LETTERS)}
IDX_TO_LETTER = {i: c for c, i in LETTER_TO_IDX.items()}

SYSTEM_INSTRUCT = (
    "당신은 이미지 기반 객관식 질의응답 모델입니다. "
    "이미지와 질문, 보기만 근거로 판단하세요. "
    "추론 과정은 출력하지 말고, 정답은 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요."
)

PROMPT_TEMPLATES = {
    0: lambda q, a, b, c, d: (
        "이미지를 보고 아래 객관식 문제를 푸세요.\n\n"
        f"질문: {q}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "정답:"
    ),
    1: lambda q, a, b, c, d: (
        "다음은 이미지 기반 4지선다 문제입니다.\n"
        "이미지에 보이는 내용만 근거로 가장 적절한 보기를 고르세요.\n\n"
        f"[문제] {q}\n"
        f"[보기 a] {a}\n"
        f"[보기 b] {b}\n"
        f"[보기 c] {c}\n"
        f"[보기 d] {d}\n\n"
        "답:"
    ),
}

def safe_text(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()

def resolve_image_path(path_str: str) -> str:
    path_str = str(path_str)
    if os.path.exists(path_str):
        return path_str
    alt = os.path.join("/content", path_str.lstrip("./"))
    if os.path.exists(alt):
        return alt
    return path_str

def row_to_dict(row_like: Any) -> Dict[str, Any]:
    if isinstance(row_like, dict):
        return dict(row_like)
    if hasattr(row_like, "to_dict"):
        return dict(row_like.to_dict())
    return dict(row_like)

def build_mc_prompt(row: Dict[str, Any], prompt_variant: int = 0) -> str:
    fn = PROMPT_TEMPLATES[prompt_variant]
    return fn(
        safe_text(row["question"]),
        safe_text(row["a"]),
        safe_text(row["b"]),
        safe_text(row["c"]),
        safe_text(row["d"]),
    )

def build_messages_from_row(
    row_like: Any,
    prompt_variant: int = 0,
    assistant_answer: Optional[str] = None,
):
    row = row_to_dict(row_like)
    img_path = resolve_image_path(row["path"])
    img = Image.open(img_path).convert("RGB")

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": build_mc_prompt(row, prompt_variant=prompt_variant)},
            ],
        },
    ]

    if assistant_answer is not None:
        messages.append(
            {"role": "assistant", "content": [{"type": "text", "text": str(assistant_answer)}]}
        )

    return messages, img

def apply_chat_template_compat(
    processor,
    messages,
    tokenize: bool = False,
    add_generation_prompt: bool = False,
    enable_thinking: bool = False,
):
    kwargs = dict(
        tokenize=tokenize,
        add_generation_prompt=add_generation_prompt,
    )
    try:
        return processor.apply_chat_template(
            messages,
            enable_thinking=enable_thinking,
            **kwargs,
        )
    except TypeError:
        return processor.apply_chat_template(messages, **kwargs)

def find_last_subsequence(seq: List[int], sub: List[int]) -> int:
    if len(sub) == 0 or len(seq) < len(sub):
        return -1
    for s in range(len(seq) - len(sub), -1, -1):
        if seq[s : s + len(sub)] == sub:
            return s
    return -1

def option_permutation_seed(row_id: Any, epoch: int, salt: int = 0) -> int:
    h = hashlib.md5(f"{row_id}|{epoch}|{salt}|{SEED}".encode("utf-8")).hexdigest()
    return int(h[:8], 16)

def apply_option_permutation(row_like: Any, perm: List[int]):
    row = row_to_dict(row_like)
    out = dict(row)

    # new index -> original index
    for new_idx, orig_idx in enumerate(perm):
        out[LETTERS[new_idx]] = row[LETTERS[orig_idx]]

    orig_answer = safe_text(row["answer"]).lower()
    orig_answer_idx = LETTER_TO_IDX[orig_answer]
    new_answer_idx = perm.index(orig_answer_idx)
    out["answer"] = LETTERS[new_answer_idx]

    new_to_orig = {LETTERS[new_idx]: LETTERS[orig_idx] for new_idx, orig_idx in enumerate(perm)}
    orig_to_new = {orig: new for new, orig in new_to_orig.items()}
    return out, new_to_orig, orig_to_new

def remap_probs_original_to_current(probs_orig: np.ndarray, new_to_orig: Dict[str, str]) -> np.ndarray:
    probs_cur = np.zeros(4, dtype=np.float32)
    for new_letter, orig_letter in new_to_orig.items():
        probs_cur[LETTER_TO_IDX[new_letter]] = float(probs_orig[LETTER_TO_IDX[orig_letter]])
    return probs_cur

def remap_logits_current_to_original(logits_cur: torch.Tensor, new_to_orig: Dict[str, str]) -> torch.Tensor:
    logits_orig = torch.empty_like(logits_cur)
    for new_letter, orig_letter in new_to_orig.items():
        logits_orig[LETTER_TO_IDX[orig_letter]] = logits_cur[LETTER_TO_IDX[new_letter]]
    return logits_orig

def label_smooth_probs(probs: np.ndarray, smoothing: float) -> np.ndarray:
    if smoothing <= 0:
        return probs
    return probs * (1.0 - smoothing) + (smoothing / 4.0)

# ---------------------------------
# Dataset / Collator
# ---------------------------------
class NoiseRobustVQADataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        train: bool = True,
        option_shuffle_prob: float = 0.0,
        prompt_variant_ids: Optional[List[int]] = None,
    ):
        self.df = df.reset_index(drop=True).copy()
        self.train = train
        self.option_shuffle_prob = option_shuffle_prob
        self.prompt_variant_ids = prompt_variant_ids or [0]
        self.current_epoch = 0

    def set_epoch(self, epoch: int):
        self.current_epoch = int(epoch)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx].to_dict()
        row_id = row.get("id", idx)

        rng = random.Random(option_permutation_seed(row_id=row_id, epoch=self.current_epoch, salt=idx))

        if self.train and len(self.prompt_variant_ids) > 1:
            prompt_variant = self.prompt_variant_ids[rng.randrange(len(self.prompt_variant_ids))]
        else:
            prompt_variant = self.prompt_variant_ids[0]

        if self.train and rng.random() < self.option_shuffle_prob:
            perm = rng.sample([0, 1, 2, 3], 4)
        else:
            perm = [0, 1, 2, 3]

        row_cur, new_to_orig, _ = apply_option_permutation(row, perm)

        teacher_probs_orig = None
        teacher_cols = ["teacher_p_a", "teacher_p_b", "teacher_p_c", "teacher_p_d"]
        if all(col in row for col in teacher_cols):
            teacher_probs_orig = np.array([float(row[col]) for col in teacher_cols], dtype=np.float32)

        if teacher_probs_orig is not None:
            teacher_probs_cur = remap_probs_original_to_current(teacher_probs_orig, new_to_orig)
        else:
            teacher_probs_cur = None

        gold_letter = safe_text(row_cur["answer"]).lower()
        gold_idx = LETTER_TO_IDX[gold_letter]
        onehot = np.eye(4, dtype=np.float32)[gold_idx]

        gold_mix = float(row.get("gold_mix", 1.0))
        sample_weight = float(row.get("sample_weight", 1.0))

        if teacher_probs_cur is not None:
            target_probs = gold_mix * onehot + (1.0 - gold_mix) * teacher_probs_cur
        else:
            target_probs = onehot

        target_probs = label_smooth_probs(target_probs.astype(np.float32), LABEL_SMOOTHING)

        messages, img = build_messages_from_row(
            row_cur,
            prompt_variant=prompt_variant,
            assistant_answer=DUMMY_ASSISTANT_ANSWER,
        )

        return {
            "messages": messages,
            "image": img,
            "gold_idx": gold_idx,
            "target_probs": target_probs.astype(np.float32),
            "sample_weight": np.float32(sample_weight),
            "row_id": row_id,
        }

@dataclass
class ChoiceLogitCollator:
    processor: Any
    enable_thinking: bool = False
    dummy_answer: str = "a"

    def __post_init__(self):
        self.dummy_answer_ids = self.processor.tokenizer(
            self.dummy_answer,
            add_special_tokens=False,
        )["input_ids"]

    def __call__(self, batch: List[Dict[str, Any]]):
        texts = [
            apply_chat_template_compat(
                self.processor,
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=self.enable_thinking,
            ).strip()
            for sample in batch
        ]
        images = [sample["image"] for sample in batch]

        proc_batch = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        input_ids = proc_batch["input_ids"]
        choice_positions = []

        for i in range(len(batch)):
            seq = input_ids[i].tolist()
            answer_start = find_last_subsequence(seq, self.dummy_answer_ids)
            if answer_start <= 0:
                decoded = self.processor.tokenizer.decode(seq, skip_special_tokens=False)
                raise ValueError(
                    "Could not locate dummy answer span.\n"
                    f"dummy_answer={self.dummy_answer}\n"
                    f"dummy_answer_ids={self.dummy_answer_ids}\n"
                    f"decoded={decoded[:1500]}"
                )
            choice_positions.append(answer_start - 1)

        proc_batch["choice_positions"] = torch.tensor(choice_positions, dtype=torch.long)
        proc_batch["gold_indices"] = torch.tensor(
            [int(sample.get("gold_idx", 0)) for sample in batch],
            dtype=torch.long,
        )
        proc_batch["target_probs"] = torch.tensor(
            np.stack([sample.get("target_probs", np.full(4, 0.25, dtype=np.float32)) for sample in batch]),
            dtype=torch.float32,
        )
        proc_batch["sample_weights"] = torch.tensor(
            [float(sample.get("sample_weight", 1.0)) for sample in batch],
            dtype=torch.float32,
        )
        proc_batch["row_ids"] = [sample.get("row_id", i) for i, sample in enumerate(batch)]
        return proc_batch

# ---------------------------------
# 모델 입력 / 로짓 / loss / metrics
# ---------------------------------
META_BATCH_KEYS = {"choice_positions", "gold_indices", "target_probs", "sample_weights", "row_ids"}

def move_batch_to_device(batch: Dict[str, Any], device: torch.device):
    moved = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            moved[k] = v.to(device)
        else:
            moved[k] = v
    return moved

def extract_model_inputs(batch: Dict[str, Any]) -> Dict[str, Any]:
    return {k: v for k, v in batch.items() if k not in META_BATCH_KEYS}

def compute_choice_logits(model, batch: Dict[str, Any], choice_token_ids: torch.Tensor) -> torch.Tensor:
    model_inputs = extract_model_inputs(batch)
    outputs = model(**model_inputs)
    logits = outputs.logits

    batch_indices = torch.arange(logits.shape[0], device=logits.device)
    choice_positions = batch["choice_positions"].to(logits.device)

    position_logits = logits[batch_indices, choice_positions, :]
    choice_logits = position_logits.index_select(dim=-1, index=choice_token_ids.to(logits.device))
    return choice_logits

def compute_noise_robust_loss(choice_logits: torch.Tensor, target_probs: torch.Tensor, sample_weights: torch.Tensor):
    log_probs = F.log_softmax(choice_logits, dim=-1)
    per_sample_loss = -(target_probs * log_probs).sum(dim=-1)
    weighted_loss = (per_sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)
    return weighted_loss, per_sample_loss

def probs_from_logits(choice_logits: torch.Tensor) -> torch.Tensor:
    return F.softmax(choice_logits, dim=-1)

def summarize_logits_against_df(choice_logits: torch.Tensor, df: pd.DataFrame) -> Dict[str, float]:
    gold_idx = torch.tensor(df["answer"].map(LETTER_TO_IDX).tolist(), dtype=torch.long)
    weights = torch.tensor(df["sample_weight"].tolist(), dtype=torch.float32)
    target_probs = torch.tensor(
        df[[f"target_p_{c}" for c in LETTERS]].to_numpy(dtype=np.float32),
        dtype=torch.float32,
    )

    pred_idx = choice_logits.argmax(dim=-1).cpu()
    log_probs = choice_logits.log_softmax(dim=-1).cpu()
    per_sample_loss = -(target_probs * log_probs).sum(dim=-1)

    raw_acc = (pred_idx == gold_idx).float().mean().item()
    weighted_acc = (((pred_idx == gold_idx).float() * weights).sum() / weights.sum().clamp_min(1e-8)).item()
    weighted_loss = ((per_sample_loss * weights).sum() / weights.sum().clamp_min(1e-8)).item()

    return {
        "raw_acc": float(raw_acc),
        "weighted_acc": float(weighted_acc),
        "weighted_ce": float(weighted_loss),
    }

# ---------------------------------
# teacher scoring / evaluation utilities
# ---------------------------------
def score_df_choice_logits(
    model,
    processor,
    df: pd.DataFrame,
    prompt_variant_ids: List[int],
    choice_token_ids: torch.Tensor,
    batch_size: int = 1,
    desc: str = "score",
    disable_adapter: bool = False,
):
    collator = ChoiceLogitCollator(processor, enable_thinking=ENABLE_THINKING, dummy_answer=DUMMY_ASSISTANT_ANSWER)
    logits_sum = None

    ctx = model.disable_adapter() if disable_adapter and hasattr(model, "disable_adapter") else nullcontext()

    with ctx:
        for prompt_variant in prompt_variant_ids:
            ds = NoiseRobustVQADataset(
                df,
                train=False,
                option_shuffle_prob=0.0,
                prompt_variant_ids=[prompt_variant],
            )
            loader = DataLoader(
                ds,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collator,
                num_workers=NUM_WORKERS,
                pin_memory=torch.cuda.is_available(),
            )

            variant_logits = []
            for batch in tqdm(loader, desc=f"{desc} | prompt={prompt_variant}", unit="batch"):
                batch = move_batch_to_device(batch, target_device)
                with torch.no_grad():
                    with torch.autocast(
                        device_type="cuda",
                        dtype=TORCH_DTYPE,
                        enabled=torch.cuda.is_available(),
                    ):
                        logits = compute_choice_logits(model, batch, choice_token_ids)
                variant_logits.append(logits.float().cpu())

            variant_logits = torch.cat(variant_logits, dim=0)
            logits_sum = variant_logits if logits_sum is None else (logits_sum + variant_logits)

    return logits_sum / max(len(prompt_variant_ids), 1)

def attach_teacher_scores(df: pd.DataFrame, choice_logits: torch.Tensor) -> pd.DataFrame:
    probs = probs_from_logits(choice_logits).numpy()
    out = df.reset_index(drop=True).copy()

    for i, c in enumerate(LETTERS):
        out[f"teacher_p_{c}"] = probs[:, i]

    pred_idx = probs.argmax(axis=1)
    sorted_probs = np.sort(probs, axis=1)
    conf = sorted_probs[:, -1]
    second = sorted_probs[:, -2]
    margin = conf - second
    entropy = -(probs * np.log(probs + 1e-12)).sum(axis=1)

    out["teacher_pred"] = [IDX_TO_LETTER[int(i)] for i in pred_idx]
    out["teacher_conf"] = conf
    out["teacher_margin"] = margin
    out["teacher_entropy"] = entropy
    out["teacher_agree"] = (out["teacher_pred"] == out["answer"].astype(str).str.lower()).astype(int)

    return out

def build_noise_robust_targets(df: pd.DataFrame) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    probs = out[[f"teacher_p_{c}" for c in LETTERS]].to_numpy(dtype=np.float32)
    gold_idx = out["answer"].astype(str).str.lower().map(LETTER_TO_IDX).to_numpy()

    gold_prob = probs[np.arange(len(out)), gold_idx]
    pred_idx = probs.argmax(axis=1)
    pred_conf = probs.max(axis=1)
    second = np.sort(probs, axis=1)[:, -2]
    margin = pred_conf - second
    agree = pred_idx == gold_idx

    # gold와 teacher를 부드럽게 섞되,
    # high-confidence disagreement는 강하게 down-weight
    gold_mix = np.clip(0.15 + 0.75 * gold_prob + 0.15 * agree.astype(np.float32), 0.15, 0.95)
    sample_weight = np.clip(0.10 + 0.90 * gold_prob + 0.20 * agree.astype(np.float32), 0.10, 1.00)

    hard_noisy = (~agree) & (pred_conf >= 0.85) & (margin >= 0.20)
    sample_weight[hard_noisy] = np.minimum(sample_weight[hard_noisy], 0.20)
    gold_mix[hard_noisy] = np.minimum(gold_mix[hard_noisy], 0.25)

    out["gold_mix"] = gold_mix.astype(np.float32)
    out["sample_weight"] = sample_weight.astype(np.float32)
    out["phase1_keep"] = (sample_weight >= 0.65).astype(int)
    out["eval_keep"] = (sample_weight >= 0.30).astype(int)

    target_mat = []
    for i in range(len(out)):
        onehot = np.eye(4, dtype=np.float32)[gold_idx[i]]
        target = gold_mix[i] * onehot + (1.0 - gold_mix[i]) * probs[i]
        target = label_smooth_probs(target.astype(np.float32), LABEL_SMOOTHING)
        target_mat.append(target)

    target_mat = np.stack(target_mat)
    for i, c in enumerate(LETTERS):
        out[f"target_p_{c}"] = target_mat[:, i]

    return out

# py3.10 이하 호환용
try:
    from contextlib import nullcontext
except Exception:
    class nullcontext:
        def __enter__(self): return self
        def __exit__(self, *excinfo): return False

def save_text(path: str, text: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(str(text))

def read_text(path: str, default: str = "") -> str:
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()


# Processor / Base model / teacher cache

여기서 먼저 **base model만** 불러온 뒤,  
train/valid에 대해 **teacher choice logits**를 캐시합니다.

이 단계가 중요한 이유:
- base zero-shot이 이미 강함
- 데이터셋에 오답/애매한 샘플이 섞여 있음
- LoRA가 base를 망치지 않도록 teacher 분포로 anchor를 걸어야 함


In [ ]:

# ---------------------------------
# optional: Gemma 4 clippable linear patch
# ---------------------------------
def maybe_patch_gemma4_clippable_linear():
    """
    Gemma 4 vision tower의 Gemma4ClippableLinear가 일부 PEFT 조합에서
    LoRA 삽입 시 문제가 날 수 있어 필요 시 monkey patch.
    """
    try:
        import torch.nn as nn
        from transformers.models.gemma4 import modeling_gemma4
    except Exception as e:
        print("Patch skipped:", repr(e))
        return

    class PatchedClippableLinear(nn.Linear):
        def __init__(self, config, in_features, out_features):
            nn.Linear.__init__(self, in_features, out_features, bias=False)
            self.use_clipped_linears = getattr(config, "use_clipped_linears", False)
            if self.use_clipped_linears:
                self.register_buffer("input_min", torch.tensor(-float("inf")))
                self.register_buffer("input_max", torch.tensor(float("inf")))
                self.register_buffer("output_min", torch.tensor(-float("inf")))
                self.register_buffer("output_max", torch.tensor(float("inf")))

        def forward(self, x):
            if self.use_clipped_linears:
                x = torch.clamp(x, self.input_min, self.input_max)
            out = nn.Linear.forward(self, x)
            if self.use_clipped_linears:
                out = torch.clamp(out, self.output_min, self.output_max)
            return out

    modeling_gemma4.Gemma4ClippableLinear = PatchedClippableLinear
    print("Gemma4ClippableLinear patch applied.")

if ENABLE_VISION_TOWER_LORA:
    maybe_patch_gemma4_clippable_linear()

# ---------------------------------
# processor / quantization
# ---------------------------------
processor = AutoProcessor.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=TORCH_DTYPE,
    bnb_4bit_quant_storage=TORCH_DTYPE,
)

base_model = AutoVisionTextModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
    attn_implementation=ATTN_IMPLEMENTATION,
    low_cpu_mem_usage=True,
)

base_model.eval()
target_device = next(base_model.parameters()).device
print("Target device:", target_device)

# choice token ids
choice_token_ids_list = []
for c in LETTERS:
    ids = processor.tokenizer(c, add_special_tokens=False)["input_ids"]
    print(f"choice token {c}: {ids}")
    if len(ids) != 1:
        raise ValueError(
            f"Choice letter '{c}' is not a single token: {ids}. "
            "이 노트북은 a/b/c/d 한 글자 분류를 전제로 합니다."
        )
    choice_token_ids_list.append(ids[0])

choice_token_ids = torch.tensor(choice_token_ids_list, dtype=torch.long)

# ---------------------------------
# teacher cache
# ---------------------------------
teacher_train_cache = os.path.join(RUN_DIR, "teacher_scored_train_pool.csv")
teacher_valid_cache = os.path.join(RUN_DIR, "teacher_scored_valid.csv")

if os.path.exists(teacher_train_cache) and os.path.exists(teacher_valid_cache):
    train_pool_scored_df = pd.read_csv(teacher_train_cache)
    valid_scored_df = pd.read_csv(teacher_valid_cache)
    print("teacher cache 불러옴.")
else:
    print("teacher cache 생성 시작...")
    train_teacher_logits = score_df_choice_logits(
        base_model,
        processor,
        train_pool_df,
        prompt_variant_ids=PROMPT_VARIANT_IDS_EVAL,
        choice_token_ids=choice_token_ids,
        batch_size=TEACHER_BATCH_SIZE,
        desc="Teacher train_pool",
        disable_adapter=False,
    )
    valid_teacher_logits = score_df_choice_logits(
        base_model,
        processor,
        valid_df,
        prompt_variant_ids=PROMPT_VARIANT_IDS_EVAL,
        choice_token_ids=choice_token_ids,
        batch_size=TEACHER_BATCH_SIZE,
        desc="Teacher valid",
        disable_adapter=False,
    )

    train_pool_scored_df = build_noise_robust_targets(attach_teacher_scores(train_pool_df, train_teacher_logits))
    valid_scored_df = build_noise_robust_targets(attach_teacher_scores(valid_df, valid_teacher_logits))

    train_pool_scored_df.to_csv(teacher_train_cache, index=False)
    valid_scored_df.to_csv(teacher_valid_cache, index=False)
    print("teacher cache 저장 완료.")

clean_valid_df = valid_scored_df[valid_scored_df["eval_keep"] == 1].reset_index(drop=True)
stage1_train_df = train_pool_scored_df[train_pool_scored_df["phase1_keep"] == 1].reset_index(drop=True)

print("\n[teacher agreement stats]")
print("train_pool teacher agree ratio:", float(train_pool_scored_df["teacher_agree"].mean()))
print("valid teacher agree ratio:", float(valid_scored_df["teacher_agree"].mean()))
print("stage1_train size:", len(stage1_train_df))
print("clean_valid size:", len(clean_valid_df))

display(
    train_pool_scored_df[
        ["id", "answer", "teacher_pred", "teacher_conf", "teacher_margin", "gold_mix", "sample_weight"]
    ].head(10)
)


# LoRA model 생성

정확도 우선 기본값:
- **vision tower는 기본 freeze**
- language model attention / MLP + image projection만 LoRA
- **rsLoRA**
- 가능하면 **LoftQ init**


In [ ]:

# ---------------------------------
# base model -> kbit training 준비
# ---------------------------------
base_model.config.use_cache = False
if hasattr(base_model.config, "text_config"):
    base_model.config.text_config.use_cache = False

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# ---------------------------------
# target modules
# ---------------------------------
if ENABLE_VISION_TOWER_LORA:
    lora_target_modules = "all-linear"
else:
    # language model attn/mlp + image->text projection
    lora_target_modules = (
        r".*(language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|"
        r"mlp\.(gate_proj|up_proj|down_proj))|embed_vision\.embedding_projection)$"
    )

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=lora_target_modules,
    task_type="CAUSAL_LM",
    use_rslora=USE_RSLORA,
    ensure_weight_tying=True,
)

start_epoch = 1
best_clean_valid_acc = -1.0
best_all_valid_acc = -1.0
best_clean_valid_ce = float("inf")
patience_counter = 0
history = []

if RESUME_FROM_CHECKPOINT:
    model = PeftModel.from_pretrained(
        base_model,
        RESUME_FROM_CHECKPOINT,
        is_trainable=True,
    )
    processor = AutoProcessor.from_pretrained(RESUME_FROM_CHECKPOINT)

    trainer_state_path = os.path.join(RESUME_FROM_CHECKPOINT, "trainer_state.pt")
    if os.path.exists(trainer_state_path):
        resume_state = torch.load(trainer_state_path, map_location="cpu")
        start_epoch = int(resume_state.get("epoch", 0)) + 1
        best_clean_valid_acc = float(resume_state.get("best_clean_valid_acc", -1.0))
        best_all_valid_acc = float(resume_state.get("best_all_valid_acc", -1.0))
        best_clean_valid_ce = float(resume_state.get("best_clean_valid_ce", float("inf")))
        patience_counter = int(resume_state.get("patience_counter", 0))
        history = resume_state.get("history", [])
    print("Resume from:", RESUME_FROM_CHECKPOINT)
else:
    model = get_peft_model(base_model, lora_config)

    if USE_LOFTQ_INIT and replace_lora_weights_loftq is not None:
        try:
            replace_lora_weights_loftq(model)
            print("LoftQ init applied.")
        except Exception as e:
            print("LoftQ init skipped:", repr(e))

model.print_trainable_parameters()
target_device = next((p.device for p in model.parameters() if p.requires_grad), next(model.parameters()).device)
print("Training target device:", target_device)
print("start_epoch:", start_epoch)


# DataLoader / phase plan

학습은 두 단계로 진행합니다.

1. **stage1 clean warm-up**  
   teacher 기준 상대적으로 깨끗한 subset으로 짧게 적응

2. **stage2 full weighted**  
   전체 train pool을 쓰되, noisy sample은 weight를 낮게 반영


In [ ]:

# ---------------------------------
# phase plan
# ---------------------------------
PHASE_PLAN = (
    [("stage1_clean_warmup", stage1_train_df)] * NUM_EPOCHS_STAGE1
    + [("stage2_full_weighted", train_pool_scored_df)] * NUM_EPOCHS_STAGE2
)
TOTAL_EPOCHS = len(PHASE_PLAN)

def build_train_loader(df: pd.DataFrame, epoch: int):
    ds = NoiseRobustVQADataset(
        df,
        train=True,
        option_shuffle_prob=TRAIN_OPTION_SHUFFLE_PROB,
        prompt_variant_ids=PROMPT_VARIANT_IDS_TRAIN,
    )
    ds.set_epoch(epoch)

    g = torch.Generator()
    g.manual_seed(SEED + epoch)

    loader = DataLoader(
        ds,
        batch_size=PER_DEVICE_BATCH_SIZE,
        shuffle=True,
        generator=g,
        collate_fn=ChoiceLogitCollator(processor, enable_thinking=ENABLE_THINKING, dummy_answer=DUMMY_ASSISTANT_ANSWER),
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return ds, loader

# optimizer steps
total_optimizer_steps = 0
for _, phase_df in PHASE_PLAN:
    phase_batches = math.ceil(len(phase_df) / PER_DEVICE_BATCH_SIZE)
    total_optimizer_steps += math.ceil(phase_batches / GRAD_ACCUM)

print("TOTAL_EPOCHS:", TOTAL_EPOCHS)
print("total_optimizer_steps:", total_optimizer_steps)

# optimizer / scheduler
adamw_kwargs = dict(lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
try:
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        fused=torch.cuda.is_available(),
        **adamw_kwargs,
    )
except TypeError:
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        **adamw_kwargs,
    )

num_warmup_steps = max(1, int(total_optimizer_steps * WARMUP_RATIO))
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max(total_optimizer_steps, 1),
)

scaler = torch.cuda.amp.GradScaler(enabled=(torch.cuda.is_available() and TORCH_DTYPE == torch.float16))

if RESUME_FROM_CHECKPOINT:
    trainer_state_path = os.path.join(RESUME_FROM_CHECKPOINT, "trainer_state.pt")
    if os.path.exists(trainer_state_path):
        resume_state = torch.load(trainer_state_path, map_location="cpu")
        if resume_state.get("optimizer_state_dict") is not None:
            optimizer.load_state_dict(resume_state["optimizer_state_dict"])
        if resume_state.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(resume_state["scheduler_state_dict"])
        if resume_state.get("scaler_state_dict") is not None and scaler.is_enabled():
            scaler.load_state_dict(resume_state["scaler_state_dict"])

best_adapter_dir = os.path.join(RUN_DIR, "best_adapter")
final_adapter_dir = os.path.join(RUN_DIR, "final_adapter")

print("best_adapter_dir:", best_adapter_dir)
print("final_adapter_dir:", final_adapter_dir)


# Fine-tuning

체크포인트 선정 기준:
1. `clean_valid_acc`
2. 동률이면 `all_valid_acc`
3. 동률이면 `clean_valid_ce` 감소

즉, **노이즈가 덜한 valid subset의 accuracy**를 최우선으로 봅니다.


In [ ]:

def save_checkpoint(
    epoch: int,
    model,
    processor,
    optimizer,
    scheduler,
    scaler,
    history: List[Dict[str, Any]],
    best_clean_valid_acc: float,
    best_all_valid_acc: float,
    best_clean_valid_ce: float,
    patience_counter: int,
):
    ckpt_dir = os.path.join(RUN_DIR, f"checkpoint-epoch-{epoch:02d}")
    os.makedirs(ckpt_dir, exist_ok=True)

    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)

    trainer_state = {
        "epoch": epoch,
        "best_clean_valid_acc": best_clean_valid_acc,
        "best_all_valid_acc": best_all_valid_acc,
        "best_clean_valid_ce": best_clean_valid_ce,
        "patience_counter": patience_counter,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "history": history,
        "config": {
            "MODEL_ID": MODEL_ID,
            "ENABLE_THINKING": ENABLE_THINKING,
            "VALID_RATIO": VALID_RATIO,
            "PER_DEVICE_BATCH_SIZE": PER_DEVICE_BATCH_SIZE,
            "GRAD_ACCUM": GRAD_ACCUM,
            "LEARNING_RATE": LEARNING_RATE,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "WARMUP_RATIO": WARMUP_RATIO,
            "MAX_GRAD_NORM": MAX_GRAD_NORM,
            "LORA_R": LORA_R,
            "LORA_ALPHA": LORA_ALPHA,
            "LORA_DROPOUT": LORA_DROPOUT,
            "USE_RSLORA": USE_RSLORA,
            "USE_LOFTQ_INIT": USE_LOFTQ_INIT,
            "ENABLE_VISION_TOWER_LORA": ENABLE_VISION_TOWER_LORA,
            "NUM_EPOCHS_STAGE1": NUM_EPOCHS_STAGE1,
            "NUM_EPOCHS_STAGE2": NUM_EPOCHS_STAGE2,
            "PROMPT_VARIANT_IDS_TRAIN": PROMPT_VARIANT_IDS_TRAIN,
            "PROMPT_VARIANT_IDS_EVAL": PROMPT_VARIANT_IDS_EVAL,
            "PROMPT_VARIANT_IDS_INFER": PROMPT_VARIANT_IDS_INFER,
            "TRAIN_OPTION_SHUFFLE_PROB": TRAIN_OPTION_SHUFFLE_PROB,
            "LABEL_SMOOTHING": LABEL_SMOOTHING,
            "SEED": SEED,
        },
    }
    torch.save(trainer_state, os.path.join(ckpt_dir, "trainer_state.pt"))

    save_text(os.path.join(RUN_DIR, "latest_checkpoint.txt"), ckpt_dir)
    pd.DataFrame(history).to_csv(os.path.join(RUN_DIR, "train_history.csv"), index=False)
    return ckpt_dir

def is_better_checkpoint(
    clean_valid_acc: float,
    all_valid_acc: float,
    clean_valid_ce: float,
    best_clean_valid_acc: float,
    best_all_valid_acc: float,
    best_clean_valid_ce: float,
):
    if clean_valid_acc > best_clean_valid_acc + EARLY_STOP_MIN_DELTA:
        return True
    if abs(clean_valid_acc - best_clean_valid_acc) <= EARLY_STOP_MIN_DELTA:
        if all_valid_acc > best_all_valid_acc + EARLY_STOP_MIN_DELTA:
            return True
        if abs(all_valid_acc - best_all_valid_acc) <= EARLY_STOP_MIN_DELTA:
            if clean_valid_ce < best_clean_valid_ce - EARLY_STOP_MIN_DELTA:
                return True
    return False

for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    phase_name, phase_df = PHASE_PLAN[epoch - 1]
    print(f"\n========== Epoch {epoch}/{TOTAL_EPOCHS} | {phase_name} ==========")

    _, train_loader = build_train_loader(phase_df, epoch=epoch)

    model.train()
    optimizer.zero_grad(set_to_none=True)

    train_loss_sum = 0.0
    train_raw_acc_sum = 0.0
    train_weight_sum = 0.0
    train_batches = 0
    grad_accum_counter = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} [train]", unit="batch")

    for step, batch in enumerate(train_bar, start=1):
        batch = move_batch_to_device(batch, target_device)

        with torch.autocast(
            device_type="cuda",
            dtype=TORCH_DTYPE,
            enabled=torch.cuda.is_available(),
        ):
            choice_logits = compute_choice_logits(model, batch, choice_token_ids)
            loss, per_sample_loss = compute_noise_robust_loss(
                choice_logits=choice_logits,
                target_probs=batch["target_probs"],
                sample_weights=batch["sample_weights"],
            )

        loss_to_backward = loss / GRAD_ACCUM

        if scaler.is_enabled():
            scaler.scale(loss_to_backward).backward()
        else:
            loss_to_backward.backward()

        with torch.no_grad():
            pred = choice_logits.argmax(dim=-1)
            correct = (pred == batch["gold_indices"]).float()
            weighted_correct = (correct * batch["sample_weights"]).sum().item()
            weight_sum = batch["sample_weights"].sum().item()

        train_loss_sum += float(loss.item())
        train_raw_acc_sum += weighted_correct
        train_weight_sum += weight_sum
        train_batches += 1
        grad_accum_counter += 1

        should_step = (grad_accum_counter == GRAD_ACCUM) or (step == len(train_loader))
        if should_step:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=MAX_GRAD_NORM,
            )

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            grad_accum_counter = 0

        train_bar.set_postfix({
            "loss": f"{train_loss_sum / max(train_batches, 1):.4f}",
            "w_acc": f"{train_raw_acc_sum / max(train_weight_sum, 1e-8):.4f}",
        })

    avg_train_loss = train_loss_sum / max(train_batches, 1)
    avg_train_w_acc = train_raw_acc_sum / max(train_weight_sum, 1e-8)

    # -----------------------------
    # validation (full once -> clean/all 둘 다 계산)
    # -----------------------------
    model.eval()

    valid_all_logits = score_df_choice_logits(
        model,
        processor,
        valid_scored_df,
        prompt_variant_ids=PROMPT_VARIANT_IDS_EVAL,
        choice_token_ids=choice_token_ids,
        batch_size=1,
        desc=f"Epoch {epoch} valid_all",
        disable_adapter=False,
    )
    all_metrics = summarize_logits_against_df(valid_all_logits, valid_scored_df)

    clean_mask = valid_scored_df["eval_keep"].to_numpy().astype(bool)
    clean_valid_logits = valid_all_logits[clean_mask]
    clean_metrics = summarize_logits_against_df(clean_valid_logits, clean_valid_df)

    clean_valid_acc = clean_metrics["raw_acc"]
    all_valid_acc = all_metrics["raw_acc"]
    clean_valid_ce = clean_metrics["weighted_ce"]

    improved = is_better_checkpoint(
        clean_valid_acc=clean_valid_acc,
        all_valid_acc=all_valid_acc,
        clean_valid_ce=clean_valid_ce,
        best_clean_valid_acc=best_clean_valid_acc,
        best_all_valid_acc=best_all_valid_acc,
        best_clean_valid_ce=best_clean_valid_ce,
    )

    if improved:
        best_clean_valid_acc = clean_valid_acc
        best_all_valid_acc = all_valid_acc
        best_clean_valid_ce = clean_valid_ce
        patience_counter = 0

        os.makedirs(best_adapter_dir, exist_ok=True)
        model.save_pretrained(best_adapter_dir)
        processor.save_pretrained(best_adapter_dir)
        save_text(os.path.join(RUN_DIR, "best_checkpoint.txt"), os.path.join(RUN_DIR, f"checkpoint-epoch-{epoch:02d}"))
    else:
        patience_counter += 1

    history.append({
        "epoch": epoch,
        "phase_name": phase_name,
        "train_loss": avg_train_loss,
        "train_weighted_acc": avg_train_w_acc,
        "clean_valid_acc": clean_valid_acc,
        "clean_valid_weighted_ce": clean_valid_ce,
        "all_valid_acc": all_valid_acc,
        "all_valid_weighted_ce": all_metrics["weighted_ce"],
        "best_clean_valid_acc": best_clean_valid_acc,
        "best_all_valid_acc": best_all_valid_acc,
        "best_clean_valid_ce": best_clean_valid_ce,
        "patience_counter": patience_counter,
        "train_size": len(phase_df),
    })

    ckpt_dir = save_checkpoint(
        epoch=epoch,
        model=model,
        processor=processor,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        history=history,
        best_clean_valid_acc=best_clean_valid_acc,
        best_all_valid_acc=best_all_valid_acc,
        best_clean_valid_ce=best_clean_valid_ce,
        patience_counter=patience_counter,
    )

    print(
        f"[Epoch {epoch}] "
        f"phase={phase_name} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"train_w_acc={avg_train_w_acc:.4f} | "
        f"clean_valid_acc={clean_valid_acc:.4f} | "
        f"all_valid_acc={all_valid_acc:.4f} | "
        f"clean_valid_ce={clean_valid_ce:.4f} | "
        f"checkpoint={ckpt_dir}"
    )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

os.makedirs(final_adapter_dir, exist_ok=True)
model.save_pretrained(final_adapter_dir)
processor.save_pretrained(final_adapter_dir)

print("Best adapter:", best_adapter_dir)
print("Final adapter:", final_adapter_dir)
print("Best checkpoint:", read_text(os.path.join(RUN_DIR, "best_checkpoint.txt")))
print("Latest checkpoint:", read_text(os.path.join(RUN_DIR, "latest_checkpoint.txt")))


# Inference

기본 동작:
- `best_adapter` 로드
- **prompt ensemble**
- **보기 순서 ensemble**
- 옵션으로 **base/adaptor logits blend**

속도보다 정확도를 우선한 설정입니다.


In [ ]:

def build_inference_samples_for_row(
    row_like: Any,
    prompt_variant_ids: List[int],
    num_option_permutations: int = 1,
):
    row = row_to_dict(row_like)

    perms: List[Tuple[int, int, int, int]] = [(0, 1, 2, 3)]
    if num_option_permutations > 1:
        rng = random.Random(option_permutation_seed(row_id=row.get("id", 0), epoch=99991, salt=17))
        seen = {perms[0]}
        while len(perms) < num_option_permutations:
            perm = tuple(rng.sample([0, 1, 2, 3], 4))
            if perm not in seen:
                perms.append(perm)
                seen.add(perm)

    samples = []
    for prompt_variant in prompt_variant_ids:
        for perm in perms:
            row_cur, new_to_orig, _ = apply_option_permutation(row, list(perm))
            messages, img = build_messages_from_row(
                row_cur,
                prompt_variant=prompt_variant,
                assistant_answer=DUMMY_ASSISTANT_ANSWER,
            )
            samples.append({
                "messages": messages,
                "image": img,
                "gold_idx": 0,
                "target_probs": np.full(4, 0.25, dtype=np.float32),
                "sample_weight": 1.0,
                "row_id": row.get("id", 0),
                "new_to_orig": new_to_orig,
            })
    return samples

def score_inference_samples(model, processor, samples: List[Dict[str, Any]], choice_token_ids: torch.Tensor):
    collator = ChoiceLogitCollator(processor, enable_thinking=ENABLE_THINKING, dummy_answer=DUMMY_ASSISTANT_ANSWER)

    logits_out = []
    for start in range(0, len(samples), INFER_VARIANT_BATCH_SIZE):
        chunk = samples[start : start + INFER_VARIANT_BATCH_SIZE]
        batch = collator(chunk)
        batch = move_batch_to_device(batch, target_device)

        with torch.no_grad():
            with torch.autocast(
                device_type="cuda",
                dtype=TORCH_DTYPE,
                enabled=torch.cuda.is_available(),
            ):
                logits = compute_choice_logits(model, batch, choice_token_ids)

        logits_out.extend([x.float().cpu() for x in logits])

    return logits_out

def predict_row_with_ensemble(
    model,
    processor,
    row_like: Any,
    choice_token_ids: torch.Tensor,
    prompt_variant_ids: List[int],
    num_option_permutations: int,
    enable_base_blend: bool = True,
    base_blend_weight: float = 0.35,
):
    samples = build_inference_samples_for_row(
        row_like,
        prompt_variant_ids=prompt_variant_ids,
        num_option_permutations=num_option_permutations,
    )

    adapter_logits = score_inference_samples(model, processor, samples, choice_token_ids)

    if enable_base_blend and hasattr(model, "disable_adapter"):
        with model.disable_adapter():
            base_logits = score_inference_samples(model, processor, samples, choice_token_ids)
    else:
        base_logits = [None] * len(adapter_logits)

    mapped_logits = []
    for i, sample in enumerate(samples):
        cur_logits = adapter_logits[i]
        if base_logits[i] is not None:
            cur_logits = (1.0 - base_blend_weight) * cur_logits + base_blend_weight * base_logits[i]

        cur_logits = remap_logits_current_to_original(cur_logits, sample["new_to_orig"])
        mapped_logits.append(cur_logits)

    final_logits = torch.stack(mapped_logits, dim=0).mean(dim=0)
    pred_idx = int(final_logits.argmax().item())
    pred_letter = IDX_TO_LETTER[pred_idx]
    pred_probs = F.softmax(final_logits, dim=-1).numpy()

    return {
        "pred_letter": pred_letter,
        "logits": final_logits,
        "probs": pred_probs,
    }

ADAPTER_FOR_INFERENCE = os.path.join(RUN_DIR, "best_adapter")
if not os.path.exists(ADAPTER_FOR_INFERENCE):
    ADAPTER_FOR_INFERENCE = best_adapter_dir if os.path.exists(best_adapter_dir) else final_adapter_dir

try:
    del model
    del base_model
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if ADAPTER_FOR_INFERENCE is None:
    infer_processor = AutoProcessor.from_pretrained(MODEL_ID)
else:
    infer_processor = AutoProcessor.from_pretrained(ADAPTER_FOR_INFERENCE)

infer_base_model = AutoVisionTextModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
    attn_implementation=ATTN_IMPLEMENTATION,
    low_cpu_mem_usage=True,
)

if ADAPTER_FOR_INFERENCE is None:
    infer_model = infer_base_model
    loaded_name = f"{MODEL_ID} (no adapter)"
else:
    infer_model = PeftModel.from_pretrained(infer_base_model, ADAPTER_FOR_INFERENCE)
    loaded_name = ADAPTER_FOR_INFERENCE

infer_model.eval()
target_device = next(infer_model.parameters()).device

preds = []
debug_rows = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    pred = predict_row_with_ensemble(
        infer_model,
        infer_processor,
        row_like=row,
        choice_token_ids=choice_token_ids,
        prompt_variant_ids=PROMPT_VARIANT_IDS_INFER,
        num_option_permutations=INFER_NUM_OPTION_PERMUTATIONS,
        enable_base_blend=ENABLE_BASE_BLEND,
        base_blend_weight=BASE_BLEND_WEIGHT,
    )
    preds.append(pred["pred_letter"])

    if i < 5:
        debug_rows.append({
            "id": row["id"],
            "pred": pred["pred_letter"],
            "p_a": float(pred["probs"][0]),
            "p_b": float(pred["probs"][1]),
            "p_c": float(pred["probs"][2]),
            "p_d": float(pred["probs"][3]),
        })

submission_path = os.path.join(RUN_DIR, "submission.csv")
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv(submission_path, index=False)

debug_path = os.path.join(RUN_DIR, "inference_debug_head.csv")
pd.DataFrame(debug_rows).to_csv(debug_path, index=False)

print("Saved submission:", submission_path)
print("Saved debug:", debug_path)
print("Model used:", loaded_name)
display(pd.DataFrame(debug_rows))


In [ ]:

print("RUN_DIR:", RUN_DIR)
print("Best checkpoint:", read_text(os.path.join(RUN_DIR, "best_checkpoint.txt")))
print("Latest checkpoint:", read_text(os.path.join(RUN_DIR, "latest_checkpoint.txt")))
print("Best adapter dir:", os.path.join(RUN_DIR, "best_adapter"))
print("Final adapter dir:", os.path.join(RUN_DIR, "final_adapter"))
